# 12 · Capstone — the Inference Engine in action
Everything below is driven by the **self-contained `inference/` package** — a portable serving layer with *no* dependency on `src/`. It loads the trained pipeline (pickled under `src.*`) via a module-alias shim, takes typed **DTOs**, and returns typed result objects.

We demonstrate **all** capabilities through one engine:
1. default prediction (+ reasons, recourse, risk-based price)  ·  batch scoring
2. customer segmentation  ·  3. profit optimization  ·  4. fraud/anomaly  ·  5. drift monitoring

In [1]:
# This notebook uses ONLY the self-contained `inference` package (no `src`).
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60)

## Configure & load the engine
`InferenceConfig` points at the model + metadata (and, optionally, a separate preprocessing artifact). The engine reads the decision threshold and reference profile from metadata.

In [2]:
from inference import (InferenceConfig, LoanInferenceEngine,
                       ApplicantDTO, PredictionOptions)
engine = LoanInferenceEngine(InferenceConfig(
    model_path='../models/best_model.joblib',
    metadata_path='../models/model_metadata.json'))
engine

LoanInferenceEngine(model='stacking', threshold=0.789, preprocessor=False)

## 1 · Predict a single applicant (typed DTO in → typed result out)
Callers build an `ApplicantDTO` (not a raw dict). With options we ask for adverse-action **reasons**, **recourse**, and a risk-based **price** in the response.

In [3]:
applicant = ApplicantDTO(
    loan_amount=52000, salary=2400, outstanding_balance=71000, interest_rate=0.24,
    number_of_defaults=2, remaining_term=66, age=27, gender='male', job='Data Analyst',
    location='Beitbridge', marital_status='single', is_employed=True,
    disbursement_date='2023 09 12', applicant_id='APP-1001')
opts = PredictionOptions(include_reasons=True, include_recourse=True, include_pricing=True)
res = engine.predict(applicant, opts)
print(json.dumps(res.to_dict(), indent=2, default=str))

{
  "applicant_id": "APP-1001",
  "default_probability": 0.2005,
  "decision": "APPROVE",
  "risk_band": "Medium",
  "threshold": 0.7887716719866072,
  "pricing": {
    "suggested_interest_rate": 0.2704,
    "base_rate": 0.12
  }
}


## Find a genuinely declined applicant from real data, and explain it
We score real rows, take the highest-risk one, and produce the full explainable response (reasons + actionable recourse).

In [4]:
raw = pd.read_csv('../dataset/global_company.csv').drop(columns=['Loan Status'])
sample = raw.sample(500, random_state=11).reset_index(drop=True)
proba = engine.predict_proba(sample)
row = sample.iloc[[int(np.argmax(proba))]]
res = engine.predict(row, PredictionOptions(include_reasons=True, include_recourse=True, include_pricing=True))
print('Decision:', res.decision, '| default prob:', res.default_probability, '| band:', res.risk_band)
print('\nWhy (adverse-action reasons):')
for r in (res.reasons or []):
    print(f'  - {r.description}: {r.applicant_value} vs typical approved {r.typical_approved_value} '
          f'(risk +{r.risk_contribution})')
print('\nRecourse:', res.recourse_message)

Decision: DECLINE | default prob: 0.9884 | band: High

Why (adverse-action reasons):
  - Requested / current loan amount: 5000.0 vs typical approved 32000.0 (risk +0.3133)
  - Interest rate on the loan: 0.18 vs typical approved 0.21 (risk +0.101)
  - Occupation: Lawyer vs typical approved Engineer (risk +0.0632)
  - Declared salary / income: 1875.09 vs typical approved 2678.74 (risk +0.0137)

Recourse: To reach approval: increase Requested / current loan amount from 5000.0 to 23900.0.


## Batch scoring
`batch_predict` accepts a list of DTOs / dicts / a DataFrame and returns one result per row.

In [5]:
batch = engine.batch_predict(sample.head(8), PredictionOptions(include_reasons=False, include_pricing=True))
pd.DataFrame([{'id': r.applicant_id, 'prob': r.default_probability, 'decision': r.decision,
               'band': r.risk_band, 'suggested_rate': r.pricing['suggested_interest_rate']}
              for r in batch])

,id,prob,decision,band,suggested_rate
0,None,0.3423,APPROVE,Medium,0.3200
1,None,0.5181,APPROVE,High,0.3200
2,None,0.1120,APPROVE,Low,0.1957
3,None,0.1386,APPROVE,Low,0.2165
4,None,0.2329,APPROVE,Medium,0.3022
5,None,0.1218,APPROVE,Low,0.2032
6,None,0.1508,APPROVE,Medium,0.2266
7,None,0.1142,APPROVE,Low,0.1974


## 2 · Customer segmentation (personas by risk & value)

In [6]:
seg = engine.segment_portfolio(raw.sample(15000, random_state=1), k=4)
print('segments:', seg.n_segments)
pd.DataFrame(seg.profile)

segments: 4


,segment,size,avg_loan,avg_salary,avg_prior_defaults,avg_revenue
0,0,4645,37721.96,2472.67,0.55,7926.50
1,1,639,34863.07,2567.87,0.44,7313.02
2,2,5325,38123.47,3465.06,0.33,8324.08
3,3,4391,14906.85,2305.73,0.43,3026.37


## 3 · Profit optimization (optimise for money, not F1)
Needs realised outcomes to evaluate profit; we pass the true labels for a held-out sample.

In [7]:
full = pd.read_csv('../dataset/global_company.csv')
samp = full.sample(15000, random_state=2).reset_index(drop=True)
y = (samp['Loan Status'].str.strip().str.lower() == 'defaulted').astype(int).to_numpy()
X = samp.drop(columns=['Loan Status'])
profit = engine.optimize_profit(X, y)
profit.to_dict()

{'best_threshold': 0.6900000000000001,
 'best_profit': 70425501.91107942,
 'best_approval_rate': 0.8537333333333333,
 'profit_at_default_threshold': 69823260.39476675,
 'profit_approve_all': 42125998.75199453,
 'uplift_vs_approve_all': 28299503.159084894}

## 4 · Fraud / anomaly detection

In [8]:
flagged = engine.detect_anomalies(raw.sample(8000, random_state=3), contamination=0.02)
print('flagged:', int(flagged['is_anomaly'].sum()))
flagged.head(6)[['loan_amount','salary','outstanding_balance','number_of_defaults','anomaly_score','is_anomaly']]

flagged: 160


,loan_amount,salary,outstanding_balance,number_of_defaults,anomaly_score,is_anomaly
5246,156000.0,1818.770192,60531.031068,1,0.0969,True
6441,192500.0,7714.515852,35119.760550,2,0.0856,True
784,225000.0,5378.326373,68808.620064,0,0.0749,True
1008,237000.0,5300.755888,94020.309721,0,0.0733,True
1739,108000.0,250.000000,49722.320896,0,0.0712,True
1406,192500.0,4319.999650,77198.065047,0,0.0668,True


## 5 · Drift monitoring (PSI)
Compare a reference population against a (here, deliberately shifted) current population.

In [9]:
ref = raw.iloc[:40000].copy()
cur = raw.iloc[40000:].copy(); cur['salary'] = cur['salary'] * 1.2
engine.monitor_drift(ref, cur).head(8)

,feature,psi,severity
0,salary,0.6900,major
1,log_salary,0.6900,major
2,balance_to_income,0.4796,major
3,loan_to_income,0.2045,moderate
4,log_loan_to_income,0.2045,moderate
5,dti,0.1258,moderate
6,lti_x_defaults,0.1010,moderate
7,dti_x_defaults,0.0724,stable


---
**One engine, every capability.** The `inference/` folder + a trained `.joblib` + its metadata is fully portable — drop it into any service (FastAPI, batch job, the Streamlit app in `app/`) and call `predict` / `batch_predict` / `segment_portfolio` / `optimize_profit` / `detect_anomalies` / `monitor_drift`.